In [4]:
# 1. Installiamo le librerie necessarie
# transformers: contiene il modello CLIP pre-addestrato
# torch: serve per far girare il modello
!pip install transformers torch pandas pillow tqdm

# 2. Colleghiamo Google Drive al notebook
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f"Errore durante il collegamento a Google Drive: {e}")
    print("Assicurati di aver autorizzato l'accesso a Google Drive e riprova.")

# Importiamo gli strumenti necessari (spostato qui per risolvere l'errore)
import os
import torch
import pandas as pd
import random
from PIL import Image, ImageEnhance
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm # Serve per vedere la barra di progresso

# Base path
base_path = "/content/drive/MyDrive/ColabNotebooks"
found_files = []

print(f"Scanning {base_path} for ANY .ipynb file...\n")

if os.path.exists(base_path):
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.endswith(".ipynb"):
                full_path = os.path.join(root, file)
                found_files.append(full_path)
                print(f"Found notebook: {file}")
else:
    print(f"Directory {base_path} does not exist.")

# Helper function to read code
def extract_code_snippets(notebook_path, keywords):
    print(f"\n--- Analyzing {os.path.basename(notebook_path)} ---")
    try:
        with open(notebook_path, 'r', encoding='utf-8') as f:
            nb = json.load(f)

        for cell in nb['cells']:
            if cell['cell_type'] == 'code':
                source = "".join(cell['source'])
                # Check if any keyword is in the source code
                if any(k in source for k in keywords):
                    print(f"\n[Match found in {os.path.basename(notebook_path)}]:\n{source[:500]}..." )
    except Exception as e:
        print(f"Error reading {notebook_path}: {e}")

print("Tutto installato e collegato!")

Mounted at /content/drive
Scanning /content/drive/MyDrive/ColabNotebooks for ANY .ipynb file...

Found notebook: Tesi_DeepLearning.ipynb
Found notebook: Tesi_Parte2ML.ipynb
Found notebook: Tesi_Demo_Riconoscimento.ipynb
Tutto installato e collegato!


In [5]:
# Carichiamo il modello CLIP pre-addestrato da OpenAI
# Questo è il "cervello" che ha già visto milioni di immagini
print("Caricamento del modello CLIP in corso...")
model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

# Spostiamo il modello sulla GPU per andare veloci
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Modello caricato su: {device}")

def applica_augmentation(image):
    """
    Applica trasformazioni casuali all'immagine per data augmentation
    Restituisce una lista con: [immagine_originale, versione_augmentata]
    """
    augmented_images = [image]  # Sempre includi l'originale

    # Con 70% di probabilità, crea una versione augmentata
    if random.random() > 0.3:
        img_aug = image.copy()

        # 1. Flip orizzontale (50% probabilità)
        if random.random() > 0.5:
            img_aug = img_aug.transpose(Image.FLIP_LEFT_RIGHT)

        # 2. Rotazione leggera tra -15 e +15 gradi
        angle = random.uniform(-15, 15)
        img_aug = img_aug.rotate(angle, fillcolor=(128, 128, 128))

        # 3. Regola luminosità (fattore tra 0.7 e 1.3)
        enhancer = ImageEnhance.Brightness(img_aug)
        brightness_factor = random.uniform(0.7, 1.3)
        img_aug = enhancer.enhance(brightness_factor)

        # 4. Regola contrasto (fattore tra 0.8 e 1.2)
        enhancer = ImageEnhance.Contrast(img_aug)
        contrast_factor = random.uniform(0.8, 1.2)
        img_aug = enhancer.enhance(contrast_factor)

        augmented_images.append(img_aug)

    return augmented_images


def estraifeatures(image_path):
    """
    Prende un'immagine e restituisce una lista di numeri (vettore)
    """
    try:
        # Apre l'immagine e la converte in RGB (evita errori con PNG trasparenti)
        image = Image.open(image_path).convert("RGB")

        # Prepara l'immagine per il modello
        inputs = processor(images=image, return_tensors="pt").to(device)

        # Calcola le features (senza calcolare i gradienti, così risparmiamo memoria)
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)

        # Converte il risultato da formato GPU a una semplice lista di numeri Python
        return image_features.pooler_output.cpu().detach().numpy().flatten()
    except Exception as e:
        print(f"Errore con l'immagine {image_path}: {e}")
        return None


Caricamento del modello CLIP in corso...


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modello caricato su: cuda


In [6]:
# Definisci dove sono le tue cartelle su Drive
# ATTENZIONE: Se hai chiamato la cartella diversamente, modifica qui sotto
BASE_PATH = "/content/drive/MyDrive/Tesi"
EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp')

def count_images_in_folder(folder_path):
    """Conta solo i file immagine validi in una cartella."""
    if not os.path.exists(folder_path):
        return 0
    return len([
        f for f in os.listdir(folder_path)
        if f.lower().endswith(EXTENSIONS)
    ])

# Scansiona tutte le sottocartelle della base
print("=" * 50)
print("📊 CONTEGGIO IMMAGINI NEL DATASET")
print("=" * 50)

total_generale = 0

for main_folder in sorted(os.listdir(BASE_PATH)):
    main_path = os.path.join(BASE_PATH, main_folder)

    # Considera solo cartelle (ignora file .pkl, .csv ecc.)
    if not os.path.isdir(main_path):
        continue

    subfolders = [
        f for f in os.listdir(main_path)
        if os.path.isdir(os.path.join(main_path, f))
    ]

    # Se ha sottocartelle (es. REALE, AI)
    if subfolders:
        print(f"\n📁 {main_folder}/")
        total_cartella = 0
        for sub in sorted(subfolders):
            sub_path = os.path.join(main_path, sub)
            count = count_images_in_folder(sub_path)
            total_cartella += count
            print(f"   └── {sub:<15} {count:>5} immagini")
        print(f"   {'TOTALE':<18} {total_cartella:>5} immagini")
        total_generale += total_cartella

    # Se non ha sottocartelle ma contiene immagini direttamente
    else:
        count = count_images_in_folder(main_path)
        if count > 0:
            print(f"\n📁 {main_folder}/")
            print(f"   └── (radice)         {count:>5} immagini")
            total_generale += count

print("\n" + "=" * 50)
print(f"🗂️  TOTALE GENERALE: {total_generale} immagini")
print("=" * 50)

folder_real = '/content/drive/MyDrive/Tesi/Tesi_Images/Real'
folder_ai = '/content/drive/MyDrive/Tesi/Tesi_Images/AI'

data = [] # Qui accumuleremo tutti i dati

print("Inizio estrazione immagini REALI...")
# Legge le immagini reali
for filename in os.listdir(folder_real):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
        path = os.path.join(folder_real, filename)
        features = estraifeatures(path)
        if features is not None:
            # Aggiungiamo le features e l'etichetta (0 per Reale)
            row = {'label': 0, 'filename': filename}
            # Aggiungiamo ogni singolo numero del vettore come colonna separata
            for i, f in enumerate(features):
                row[f'feature_{i}'] = f
            data.append(row)

print("Inizio estrazione immagini AI...")
# Legge le immagini AI
for filename in os.listdir(folder_ai):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
        path = os.path.join(folder_ai, filename)
        features = estraifeatures(path)
        if features is not None:
            # Aggiungiamo le features e l'etichetta (1 per AI)
            row = {'label': 1, 'filename': filename}
            for i, f in enumerate(features):
                row[f'feature_{i}'] = f
            data.append(row)

# Creiamo il DataFrame (la tabella)
df = pd.DataFrame(data)

# Salviamo tutto in un file CSV su Drive
output_path = '/content/drive/MyDrive/Tesi/Tesi_Images/dataset_features.csv'
df.to_csv(output_path, index=False)

print(f"Finito! Ho estratto le features da {len(df)} immagini.")
print(f"Dati salvati in: {output_path}")


📊 CONTEGGIO IMMAGINI NEL DATASET

📁 ImmaginiTest/
   └── AI                547 immagini
   └── REALE             550 immagini
   TOTALE              1097 immagini

📁 Tesi_Images/
   └── AI               2077 immagini
   └── Real             2347 immagini
   TOTALE              4424 immagini

🗂️  TOTALE GENERALE: 5521 immagini
Inizio estrazione immagini REALI...
Inizio estrazione immagini AI...
Finito! Ho estratto le features da 4424 immagini.
Dati salvati in: /content/drive/MyDrive/Tesi/Tesi_Images/dataset_features.csv


In [7]:
# NUOVA CELLA: Estrai features per ImmaginiTest (holdout set)
folder_real_test = '/content/drive/MyDrive/Tesi/ImmaginiTest/REALE'
folder_ai_test   = '/content/drive/MyDrive/Tesi/ImmaginiTest/AI'
data_test = []

print("Inizio estrazione features ImmaginiTest (REALI)...")
for filename in os.listdir(folder_real_test):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
        path = os.path.join(folder_real_test, filename)
        features = estraifeatures(path)
        if features is not None:
            row = {'label': 0, 'filename': filename}
            for i, f in enumerate(features):
                row[f'feature_{i}'] = f
            data_test.append(row)

print("Inizio estrazione features ImmaginiTest (AI)...")
for filename in os.listdir(folder_ai_test):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
        path = os.path.join(folder_ai_test, filename)
        features = estraifeatures(path)
        if features is not None:
            row = {'label': 1, 'filename': filename}
            for i, f in enumerate(features):
                row[f'feature_{i}'] = f
            data_test.append(row)

df_test = pd.DataFrame(data_test)
output_test = '/content/drive/MyDrive/Tesi/ImmaginiTest/features_test.csv'
os.makedirs(os.path.dirname(output_test), exist_ok=True)
df_test.to_csv(output_test, index=False)
print(f"Finito! Estratte features da {len(df_test)} immagini di test.")
print(f"Salvato in: {output_test}")


Inizio estrazione features ImmaginiTest (REALI)...
Inizio estrazione features ImmaginiTest (AI)...
Finito! Estratte features da 1097 immagini di test.
Salvato in: /content/drive/MyDrive/Tesi/ImmaginiTest/dataset/features_test.csv
